In [1]:
import os
from dotenv import load_dotenv

load_dotenv()  # loads from .env automatically

print("OPENAI_API_KEY loaded:", bool(os.getenv("OPENAI_API_KEY")))
print("LANGSMITH loaded:", bool(os.getenv("LANGCHAIN_API_KEY")))

OPENAI_API_KEY loaded: True
LANGSMITH loaded: True


In [5]:
import os
from google.auth.transport.requests import Request
from google.oauth2.credentials import Credentials
from google_auth_oauthlib.flow import InstalledAppFlow
from googleapiclient.discovery import build

SCOPES = ["https://www.googleapis.com/auth/calendar"]

def get_calendar_service():
    creds = None

    if os.path.exists("token.json"):
        creds = Credentials.from_authorized_user_file("token.json", SCOPES)

    if not creds or not creds.valid:
        if creds and creds.expired and creds.refresh_token:
            creds.refresh(Request())
        else:
            flow = InstalledAppFlow.from_client_secrets_file("../credentials.json", SCOPES)
            creds = flow.run_local_server(port=0)

        with open("token.json", "w") as f:
            f.write(creds.to_json())

    return build("calendar", "v3", credentials=creds)


service = get_calendar_service()
print("Connected to Google Calendar!")

Please visit this URL to authorize this application: https://accounts.google.com/o/oauth2/auth?response_type=code&client_id=150244531012-6eqvbad9au91k9dhe88j7tjf95dobd3u.apps.googleusercontent.com&redirect_uri=http%3A%2F%2Flocalhost%3A55210%2F&scope=https%3A%2F%2Fwww.googleapis.com%2Fauth%2Fcalendar&state=X8Uxna5aVSUAjdbkkwy0V7CxRMjBHw&code_challenge=9UyXxOOoJc1x6urtMVG1uvPeU8WLr4R_JqIf2NAT8Cs&code_challenge_method=S256&access_type=offline
Connected to Google Calendar!


In [3]:
print(os.getcwd())

/Users/pranavr/Desktop/Pranav/Miscellaneous/Calendar_Agent/notebooks


In [6]:
from langchain_core.tools import tool
from datetime import datetime

@tool
def get_calendar_events(date: str) -> str:
    """
    Get all calendar events for a given date.
    Always call this FIRST before scheduling anything.
    Use this to find what time slots are already taken.

    Args:
        date: "today" or a date in YYYY-MM-DD format
    """
    if date == "today":
        date = datetime.now().strftime("%Y-%m-%d")

    service = get_calendar_service()

    start = f"{date}T00:00:00Z"
    end   = f"{date}T23:59:59Z"

    result = service.events().list(
        calendarId="primary",
        timeMin=start,
        timeMax=end,
        singleEvents=True,
        orderBy="startTime"
    ).execute()

    events = result.get("items", [])

    if not events:
        return f"No events on {date}. Full day is free."

    summary = f"Events on {date}:\n"
    for e in events:
        start_time = e["start"].get("dateTime", "all-day")
        end_time   = e["end"].get("dateTime", "")
        title      = e.get("summary", "Untitled")
        if start_time != "all-day":
            start_time = start_time[11:16]
            end_time   = end_time[11:16]
            summary += f"  - {title}: {start_time} to {end_time}\n"
        else:
            summary += f"  - {title}: all day\n"

    return summary

# Test it — calls your real Google Calendar
result = get_calendar_events.invoke({"date": "today"})
print(result)

No events on 2026-03-28. Full day is free.


In [7]:
@tool
def create_calendar_event(
    title: str,
    date: str,
    start_time: str,
    end_time: str
) -> str:
    """
    Create a calendar event in Google Calendar.
    Only call this after checking get_calendar_events to confirm the slot is free.
    Never double-book an existing event.

    Args:
        title:      Name of the event e.g. "Gym", "Study session"
        date:       Date in YYYY-MM-DD format
        start_time: Start in HH:MM format e.g. "09:00"
        end_time:   End in HH:MM format e.g. "10:00"
    """
    service = get_calendar_service()

    event = {
        "summary": title,
        "start": {
            "dateTime": f"{date}T{start_time}:00",
            "timeZone": "America/New_York",
        },
        "end": {
            "dateTime": f"{date}T{end_time}:00",
            "timeZone": "America/New_York",
        },
    }

    created = service.events().insert(
        calendarId="primary",
        body=event
    ).execute()

    return f"Created '{title}' on {date} from {start_time} to {end_time}."

# Test it manually
today = datetime.now().strftime("%Y-%m-%d")
result = create_calendar_event.invoke({
    "title": "Test event - delete me",
    "date": today,
    "start_time": "08:00",
    "end_time": "08:30"
})
print(result)

Created 'Test event - delete me' on 2026-03-28 from 08:00 to 08:30.
